In [1]:
import pickle
import numpy as np


class ImageObject:
    def __init__(self, data, label):
        self.data = data
        self.label = label
        self.red = data[:, :, 0]
        self.green = data[:, :, 1]
        self.blue = data[:, :, 2]
        self.grayscale = np.dot(data[..., :3], [0.299, 0.587, 0.114])

    def __repr__(self):
        return (
            f"<ImageObject label={self.label} grayscale_shape={self.grayscale.shape}>"
        )


def load_cifar10():
    try:
        with open("../dataset_cache.pkl", "rb") as f:
            train_data, test_data = pickle.load(f)
        print("Success!")
        print(train_data[0])
        return train_data, test_data

    except FileNotFoundError:
        print("Cache file not found! Please run the setup script first.")
        return [], []

train_data = []
test_data = []
train_data, test_data = load_cifar10()

print(f"Train size: {len(train_data)}")
print(f"Test size: {len(test_data)}")
xy_train = train_data[:40000]
xy_test = train_data[40000:]

Success!
<ImageObject label=frog grayscale_shape=(32, 32)>
Train size: 50000
Test size: 300000


In [2]:
from collections import Counter
class k_nn_Classifier:
    def __init__(self, k, train_data) -> None:
        self.data = np.array([i.data.flatten() for i in train_data]).astype(np.int16)
        self.reds = np.array([i.red.flatten() for i in train_data])
        self.greens = np.array([i.green.flatten() for i in train_data])
        self.blues = np.array([i.blue.flatten() for i in train_data])
        self.grays = np.array([i.grayscale.flatten() for i in train_data])
        self.labels = np.array([i.label for i in train_data])
        self.k = k

    def __repr__(self) -> str:
        return f"Data: {len(self.data)}, Labels: {len(self.labels)} Looking at {self.k} neighbors"

    def test(self, test_data, color, metric, k = self.k):
        # Basic grayscale tester with manhattan distance
        results = []
        test_grays = np.array([i.grayscale.flatten() for i in test_data])
        test_img_data = np.array([i.data.flatten() for i in test_data]).astype(np.int16)
        test_labels = [i.label for i in test_data]
        print(
            f"Testing {len(test_data)} images against {len(self.grays)} training samples..."
        )
        gray_count, data_count = 0, 0
        for i in range(len(test_grays)):
            gray_diffs = self.grays - test_grays[i]
            img_diffs = self.data - test_img_data[i]
            gray_abs_diffs = np.abs(gray_diffs)
            img_abs_diffs = np.abs(img_diffs)
            gray_distances = np.sum(gray_abs_diffs, axis=1)
            img_distances = np.sum(img_abs_diffs, axis = 1)
            gray_k_indices = np.argpartition(gray_distances, self.k)[:self.k]
            img_k_indices = np.argpartition(img_distances, self.k)[:self.k]
            gray_k_labels = self.labels[gray_k_indices]
            img_k_labels = self.labels[img_k_indices]
            gray_predicted_label = Counter(gray_k_labels).most_common(1)[0][0]
            img_predicted_label = Counter(img_k_labels).most_common(1)[0][0]
            results.append((gray_predicted_label, img_predicted_label))
            acc_label = test_labels[i]
            if gray_predicted_label == acc_label:
                gray_count += 1
            if img_predicted_label == acc_label:
                data_count += 1
            gray_acc = (gray_count/(i+1))*100
            data_acc = (data_count/(i+1))*100
            if i % 100 == 0:
                print(
                    f"Processed {i}/{len(test_grays)} \n| Pred: {gray_predicted_label}, Accuracy: {gray_acc:2f}% \n| Pred: {img_predicted_label}, Accuracy: {data_acc:2f}% \n| Acctual {test_labels[i]}"
                )
        return results

In [4]:
classifier = k_nn_Classifier(2, xy_train)
print(classifier)
results = classifier.test(xy_test, None, None)
print(results)

Data: 40000, Labels: 40000 Looking at 2 neighbors
Testing 10000 images against 40000 training samples...
Processed 0/10000 
| Pred: deer, Accuracy: 0.000000% 
| Pred: cat, Accuracy: 0.000000% 
| Acctual automobile
Processed 100/10000 
| Pred: bird, Accuracy: 34.653465% 
| Pred: bird, Accuracy: 41.584158% 
| Acctual bird
Processed 200/10000 
| Pred: truck, Accuracy: 36.815920% 
| Pred: deer, Accuracy: 41.791045% 
| Acctual truck
Processed 300/10000 
| Pred: cat, Accuracy: 37.209302% 
| Pred: ship, Accuracy: 42.857143% 
| Acctual automobile
Processed 400/10000 
| Pred: automobile, Accuracy: 36.658354% 
| Pred: dog, Accuracy: 38.653367% 
| Acctual automobile
Processed 500/10000 
| Pred: deer, Accuracy: 35.928144% 
| Pred: automobile, Accuracy: 38.722555% 
| Acctual automobile
Processed 600/10000 
| Pred: frog, Accuracy: 35.274542% 
| Pred: airplane, Accuracy: 38.768719% 
| Acctual ship
Processed 700/10000 
| Pred: bird, Accuracy: 34.094151% 
| Pred: cat, Accuracy: 37.517832% 
| Acctual do

KeyboardInterrupt: 